# Simple RAG Application using LangChain and GEMINI

In [ ]:
# Install the necessary packages
import langchain
import langchain_google_genai

In [ ]:
# Import necessary libraries
import os
from google.colab import userdata

Initialize GeminiAI LLM

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
# Set  API key
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

# Initialize model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7
)


Initialize Embedding Model

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

Create and Embedding Documents

In [ ]:
from langchain_core.documents import Document

# Define a list of documents with content and metadata
documents = [
    Document(
        page_content="The T20 World Cup 2024 is in full swing, bringing excitement and drama to cricket fans worldwide.India's team, captained by Rohit Sharma, is preparing for a crucial match against Ireland, with standout player Jasprit Bumrah expected to play a pivotal role in their campaign.The tournament has already seen controversy, particularly concerning the pitch conditions at Nassau County International Cricket Stadium in New York, which came under fire after a low-scoring game between Sri Lanka and South Africa.",
        metadata={"source": "cricket news"},
    ),
    Document(
        page_content="The world of football is buzzing with excitement as major tournaments and league matches continue to captivate fans globally.In the UEFA Champions League, the semi-final matchups have been set, with defending champions Real Madrid set to face Manchester City, while Bayern Munich will take on Paris Saint-Germain.Both ties promise thrilling encounters, featuring some of the best talents in world football.",
        metadata={"source": "football news"},
    ),
    Document(
        page_content="As election season heats up, the latest developments reveal a highly competitive atmosphere across several key races.The presidential election has seen intense campaigning from all major candidates, with recent polls indicating a tight race.Incumbent President Jane Doe is seeking re-election on a platform of economic stability and healthcare reform, while her main rival, Senator John Smith, focuses on education and climate change initiatives.",
        metadata={"source": "election news"},
    ),
    Document(
        page_content="The AI revolution continues to transform industries and reshape the global economy.Significant advancements in artificial intelligence have led to breakthroughs in healthcare, with AI-driven diagnostics improving patient outcomes and reducing costs.Autonomous systems are becoming increasingly prevalent in logistics and transportation, enhancing efficiency and safety.",
        metadata={"source": "ai revolution news"},
    ),
]


In [ ]:
# create a vector store using the docments and embedding model
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model
)

In [ ]:
results = vectorstore.similarity_search("test match")

for result in results:
  print("-----------------------")
  print(result.page_content)
  print(result.metadata)

-----------------------
The T20 World Cup 2024 is in full swing, bringing excitement and drama to cricket fans worldwide.India's team, captained by Rohit Sharma, is preparing for a crucial match against Ireland, with standout player Jasprit Bumrah expected to play a pivotal role in their campaign.The tournament has already seen controversy, particularly concerning the pitch conditions at Nassau County International Cricket Stadium in New York, which came under fire after a low-scoring game between Sri Lanka and South Africa.
{'source': 'cricket news'}
-----------------------
The T20 World Cup 2024 is in full swing, bringing excitement and drama to cricket fans worldwide.India's team, captained by Rohit Sharma, is preparing for a crucial match against Ireland, with standout player Jasprit Bumrah expected to play a pivotal role in their campaign.The tournament has already seen controversy, particularly concerning the pitch conditions at Nassau County International Cricket Stadium in New 

In [ ]:
results = vectorstore.similarity_search("machine learning")

for result in results:
  print("-----------------------")
  print(result.page_content)
  print(result.metadata)

-----------------------
The AI revolution continues to transform industries and reshape the global economy.Significant advancements in artificial intelligence have led to breakthroughs in healthcare, with AI-driven diagnostics improving patient outcomes and reducing costs.Autonomous systems are becoming increasingly prevalent in logistics and transportation, enhancing efficiency and safety.
{'source': 'ai revolution news'}
-----------------------
The AI revolution continues to transform industries and reshape the global economy.Significant advancements in artificial intelligence have led to breakthroughs in healthcare, with AI-driven diagnostics improving patient outcomes and reducing costs.Autonomous systems are becoming increasingly prevalent in logistics and transportation, enhancing efficiency and safety.
{'source': 'ai revolution news'}
-----------------------
As election season heats up, the latest developments reveal a highly competitive atmosphere across several key races.The p

Embed Query and Perform Similarity Search by Vector

In [ ]:
# Embed a query using the embedding model
query_embedding = embedding_model.embed_query("machine learning")

query_embedding [:10]

[-0.006826791,
 0.0032323739,
 0.0036080698,
 -0.063535206,
 -0.016612697,
 0.003850432,
 0.0019643204,
 0.027494766,
 0.0015598271,
 0.012391397]

In [ ]:
# Print the length of the query embedding
len(query_embedding)

3072

In [ ]:
results = vectorstore.similarity_search_by_vector(query_embedding)

for result in results:
  print("-----------------------")
  print(result.page_content)
  print(result.metadata)

-----------------------
The AI revolution continues to transform industries and reshape the global economy.Significant advancements in artificial intelligence have led to breakthroughs in healthcare, with AI-driven diagnostics improving patient outcomes and reducing costs.Autonomous systems are becoming increasingly prevalent in logistics and transportation, enhancing efficiency and safety.
{'source': 'ai revolution news'}
-----------------------
The AI revolution continues to transform industries and reshape the global economy.Significant advancements in artificial intelligence have led to breakthroughs in healthcare, with AI-driven diagnostics improving patient outcomes and reducing costs.Autonomous systems are becoming increasingly prevalent in logistics and transportation, enhancing efficiency and safety.
{'source': 'ai revolution news'}
-----------------------
As election season heats up, the latest developments reveal a highly competitive atmosphere across several key races.The p

Create Retriever

In [ ]:
# Create a retriver from the vector
retriver = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1}
)

# Perform batch retrieval using retriever
batch_results =  retriver.batch (["machine learning", "test match"])

for result in batch_results:
  print("-----------------------")
  for doc in result: # Corrected: iterate over 'result' instead of 'batch_results'
    print(doc.page_content)
    print(doc.metadata)

-----------------------
The AI revolution continues to transform industries and reshape the global economy.Significant advancements in artificial intelligence have led to breakthroughs in healthcare, with AI-driven diagnostics improving patient outcomes and reducing costs.Autonomous systems are becoming increasingly prevalent in logistics and transportation, enhancing efficiency and safety.
{'source': 'ai revolution news'}
-----------------------
The T20 World Cup 2024 is in full swing, bringing excitement and drama to cricket fans worldwide.India's team, captained by Rohit Sharma, is preparing for a crucial match against Ireland, with standout player Jasprit Bumrah expected to play a pivotal role in their campaign.The tournament has already seen controversy, particularly concerning the pitch conditions at Nassau County International Cricket Stadium in New York, which came under fire after a low-scoring game between Sri Lanka and South Africa.
{'source': 'cricket news'}


Create Prompt Template

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
# Define a message template for the chatbot
message = """
Answer this question using the provided context only.

{question}

Context:
{context}
"""

#Create a chat prompt template from the message
prompt = ChatPromptTemplate.from_messages([("human",message)])

Chain Retriver and Prompt Template with LLM


In [ ]:
chain = {"context": retriver, "question": RunnablePassthrough()} | prompt | llm

In [ ]:
response = chain.invoke ("current state of 2024 t20 world cup")
print(response.content)

The 2024 T20 World Cup is in full swing, bringing excitement and drama to cricket fans worldwide. India's team, captained by Rohit Sharma, is preparing for a crucial match against Ireland, with Jasprit Bumrah expected to play a pivotal role. The tournament has already seen controversy regarding the pitch conditions at Nassau County International Cricket Stadium in New York, particularly after a low-scoring game between Sri Lanka and South Africa.
